# Rebuild the TechJam fine-tuned BGE catalogue cache

Run this notebook in **Google Colab with a GPU runtime**. It lets you upload the fine-tuned `model_finetuned-*.zip` and the exact `catalog.jsonl` directly, rebuilds all 50,000 normalized 768-dimensional vectors, validates them, and downloads a metadata-complete cache plus manifest.

> The output is intentionally identified as a **fine-tuned** embedding space. Do not rename it to the base-BGE cache filename unless the runtime is configured to load this same model for query embeddings.

In [ ]:
%pip -q install -U "sentence-transformers==6.0.0"

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import hashlib, json, os, re, shutil, tempfile
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

EXPECTED_ROWS = 50_000
EXPECTED_DIM = 768
PRODUCT_TEXT_VERSION = 'patch2-product-text-v1'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
print('Upload the fine-tuned model ZIP and the exact catalogue.jsonl.')
uploaded = files.upload()
uploaded_names = sorted(uploaded)
print('Uploaded:', uploaded_names)

In [ ]:
def pick_uploaded(predicate, description):
    matches = [Path(name) for name in uploaded_names if predicate(Path(name))]
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one {description}; found {matches}')
    return matches[0]

model_zip = pick_uploaded(lambda p: p.suffix.lower() == '.zip' and 'model' in p.name.lower(), 'model ZIP')
catalog_path = pick_uploaded(lambda p: p.name.lower() == 'catalog.jsonl' or p.suffix.lower() == '.jsonl', 'catalog JSONL')
print('Model ZIP:', model_zip)
print('Catalogue:', catalog_path)

In [ ]:
# Extract only after validating every archive member stays inside the model directory.
extract_root = Path(tempfile.mkdtemp(prefix='techjam_model_'))
with ZipFile(model_zip) as archive:
    members = archive.namelist()
    for member in members:
        destination = (extract_root / member).resolve()
        if not str(destination).startswith(str(extract_root.resolve()) + os.sep) and destination != extract_root.resolve():
            raise ValueError(f'Unsafe ZIP member: {member}')
    archive.extractall(extract_root)
model_dirs = [p for p in extract_root.rglob('config_sentence_transformers.json')]
if len(model_dirs) != 1:
    raise ValueError(f'Could not identify exactly one SentenceTransformer directory: {model_dirs}')
model_dir = model_dirs[0].parent
weights = model_dir / 'model.safetensors'
if not weights.exists():
    raise FileNotFoundError(weights)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

weights_sha256 = sha256_file(weights)
model_id = f'BAAI/bge-base-en-v1.5+techjam-finetuned-{weights_sha256[:16]}'
backend_id = 'bge-finetuned-techjam-v1'
embedding_space_id = f'{backend_id}:{model_id}:dimensions={EXPECTED_DIM}:normalization=l2:query=bge-search-prefix-v1'
print('Model directory:', model_dir)
print('Weights SHA-256:', weights_sha256)
print('Embedding space:', embedding_space_id)

In [ ]:
def production_product_text(product):
    title = product.get('title') or ''
    categories = ', '.join(product.get('categories') or [])
    features = '; '.join((product.get('features') or [])[:3])
    return f'Product: {title}. Categories: {categories}. Features: {features}.'.strip()

products = []
with Path(catalog_path).open(encoding='utf-8') as handle:
    for line in handle:
        if line.strip():
            products.append(json.loads(line))
if len(products) != EXPECTED_ROWS:
    raise ValueError(f'Expected {EXPECTED_ROWS:,} catalogue rows, got {len(products):,}')
ids = [str(row['parent_asin']) for row in products]
if len(set(ids)) != len(ids):
    raise ValueError('Catalogue contains duplicate parent_asin values')
texts = [production_product_text(row) for row in products]

def fingerprint_texts(values):
    digest = hashlib.sha256()
    for value in values:
        encoded = str(value).encode('utf-8')
        digest.update(len(encoded).to_bytes(8, 'big'))
        digest.update(encoded)
    return digest.hexdigest()

catalog_fingerprint = sha256_file(catalog_path)
product_text_fingerprint = fingerprint_texts(texts)
print('Rows:', len(products))
print('Catalogue SHA-256:', catalog_fingerprint)
print('Product-text fingerprint:', product_text_fingerprint)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(str(model_dir), device=device)
# The saved model contains its Normalize module; normalize_embeddings=True makes the
# contract explicit and protects against a future model/config change.
embeddings = model.encode(
    texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True, output_value='sentence_embedding',
)
embeddings = np.asarray(embeddings, dtype=np.float32)
if embeddings.shape != (EXPECTED_ROWS, EXPECTED_DIM):
    raise ValueError(f'Unexpected embedding shape: {embeddings.shape}')
norms = np.linalg.norm(embeddings, axis=1)
if not np.all(np.isfinite(embeddings)) or not np.allclose(norms, 1.0, rtol=1e-4, atol=1e-5):
    raise ValueError('Embedding matrix is non-finite or not L2-normalized')
print('Built:', embeddings.shape, embeddings.nbytes / 1024**2, 'MiB')
print('Norm range:', float(norms.min()), float(norms.max()))

In [ ]:
output_cache = Path('catalog_cache_bge-finetuned-techjam-v1.npz')
metadata = {
    'schema_version': 2,
    'backend_id': backend_id,
    'model_id': model_id,
    'embedding_space_id': embedding_space_id,
    'row_count': EXPECTED_ROWS,
    'vector_dimension': EXPECTED_DIM,
    'normalized': True,
    'product_text_version': PRODUCT_TEXT_VERSION,
    'product_text_fingerprint': product_text_fingerprint,
    'catalog_fingerprint': catalog_fingerprint,
    'query_prefix': QUERY_PREFIX,
    'model_weights_sha256': weights_sha256,
}
np.savez_compressed(
    output_cache, embeddings=embeddings, ids=np.asarray(ids, dtype=np.str_),
    metadata_json=np.asarray(json.dumps(metadata, sort_keys=True), dtype=np.str_),
)
print('Wrote:', output_cache, output_cache.stat().st_size / 1024**2, 'MiB')

In [ ]:
# Reload the written artifact and verify every structural contract used by the active loader.
with np.load(output_cache, allow_pickle=False) as data:
    required = {'embeddings', 'ids', 'metadata_json'}
    if not required.issubset(data.files):
        raise ValueError(f'Missing arrays: {required - set(data.files)}')
    reloaded = np.asarray(data['embeddings'], dtype=np.float32)
    reloaded_ids = [str(value) for value in data['ids'].tolist()]
    reloaded_metadata = json.loads(str(data['metadata_json'].item()))
assert np.array_equal(reloaded, embeddings)
assert reloaded_ids == ids
assert reloaded_metadata == metadata
print(json.dumps({
    'status': 'valid', 'shape': list(reloaded.shape),
    'ids_exact_row_order': reloaded_ids == ids,
    'metadata': reloaded_metadata,
}, indent=2))

In [ ]:
manifest = {
    'schema_version': 1,
    'cache_filename': output_cache.name,
    'cache_sha256': sha256_file(output_cache),
    'model_zip': model_zip.name,
    'model_zip_sha256': sha256_file(model_zip),
    'model_weights_sha256': weights_sha256,
    'catalog_filename': catalog_path.name,
    'catalog_sha256': catalog_fingerprint,
    'product_text_fingerprint': product_text_fingerprint,
    'backend_id': backend_id, 'model_id': model_id,
    'embedding_space_id': embedding_space_id,
    'rows': EXPECTED_ROWS, 'dimensions': EXPECTED_DIM, 'normalized': True,
    'query_prefix': QUERY_PREFIX,
}
manifest_path = Path('bge_finetuned_cache_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('Downloading cache and manifest...')
files.download(str(output_cache))
files.download(str(manifest_path))